## Ensemble usando imagens originais e geradas pelos descritores fractais

Usando K-fold e regra da soma para aprimorar as predições das redes mobillenet e efficientnet.

In [ ]:
!unzip /content/dataset2.zip -d /content/dataset

In [7]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
import random
from pathlib import Path
import os


import torch.nn.functional as F

In [8]:
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
K_FOLDS = 5

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [9]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)

print("Treinando em:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Treinando em: cuda
GPU: Tesla T4


Função para carregar o dataset que vem na estrutura
``` plaintext
dataset
    treino_e_validacao
        healthy
            F-RecPlot
                1.png
                2.png
                ...
            originais
                1.tif
                2.tif
                ...
        severe
            F-RecPlot
                ...
            originais
                ...
    testes
        healthy
            ...
        severe
            ...
```
- `dir_data:` o caminho até alguma pasta que contém healthy e severe
- `class_names:` aqui é healthy e severe
- `reshape_type:` a pasta que buscamos, 'originais' ou  'F-RecPlot'

In [10]:
def load_data_from_folders(dir_data, class_names, reshape_type):
    data_list = []
    for class_index, class_name in enumerate(class_names):
        dir_class = Path(dir_data) / class_name / reshape_type

        if dir_class.exists():
            images = list(dir_class.glob('*.*'))
            print(f"Imagens encontradas em {class_name}: {len(images)}")
            for img_path in images:
                data_list.append((str(img_path), class_index))
        else:
            print(f"AVISO: Diretório {dir_class} não encontrado!!!!")

    return data_list

Para preparar o dataset

In [11]:
class ImageDataset(Dataset):
    def __init__(self, data_list, transform=None):
        self.data = data_list
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

Transformação padrão

In [12]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

Primeiro fazemos a função que treina com um único fold, em uma sequência de dados.

In [13]:
def train_one_fold(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)

    best_acc = 0.0
    best_f1 = 0.0

    for epoch in range(EPOCHS):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            #running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels)
            total += labels.size(0)

        train_acc = correct.double() / total

        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(DEVICE)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.numpy())

        f1 = f1_score(val_labels, val_preds, average='macro')
        acc = accuracy_score(val_labels, val_preds)

        best_acc = max(best_acc, acc)
        best_f1 = max(best_f1, f1)

        print(f"Época {epoch+1}/{EPOCHS} | TrainAcc {train_acc:.4f} | ValAcc {acc:.4f} | ValF1 {f1:.4f}")

    return model, best_acc, best_f1

Para facilitar a criação dos modelos

In [14]:
def criar_modelo(backbone: str, num_classes: int, pretrained=True):
    backbone = backbone.lower()

    if backbone == "mobilenet":
        model = models.mobilenet_v2(pretrained=pretrained)
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features, num_classes
        )

    elif backbone == "efficientnet_b0":
        model = models.efficientnet_b0(pretrained=pretrained)
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features, num_classes
        )

    else:
        raise ValueError("backbone deve ser 'mobilenet' ou 'efficientnet_b0'")

    return model

Essa aqui sim faz o treino com o k folds

In [15]:
def run_kfold(dataset_path, dataset_type, class_names, backbone="mobilenet"):

    data_list = load_data_from_folders(dataset_path, class_names, dataset_type)
    print(f"Total de imagens: {len(data_list)}")

    paths = [x[0] for x in data_list]
    labels = [x[1] for x in data_list]

    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

    fold_results = []

    best_f1_global = -1
    best_model_state = None

    for fold, (train_idx, val_idx) in enumerate(skf.split(paths, labels)):
        print("\n============================")
        print(f"FOLD {fold+1}/{K_FOLDS}")
        print("============================")

        train_data = [(paths[i], labels[i]) for i in train_idx]
        val_data = [(paths[i], labels[i]) for i in val_idx]

        train_dataset = ImageDataset(train_data, transform=transform)
        val_dataset = ImageDataset(val_data, transform=transform)

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

        # modelo pré-treinado
        model = criar_modelo(
          backbone=backbone,
          num_classes=len(class_names),
          pretrained=True
        ).to(DEVICE)

        model, acc, f1 = train_one_fold(
          model, train_loader, val_loader
        )

        # guarda o melhor modelo global (por F1)
        if f1 > best_f1_global:
            best_f1_global = f1
            best_model_state = model.state_dict()

            nome_arquivo_modelo = f"melhor_modelo_{backbone}_{dataset_type}.pth"
            torch.save(model.state_dict(), nome_arquivo_modelo)

            print(f"   -> Novo melhor modelo salvo! F1: {best_f1_global:.4f}")

        fold_results.append({
            'fold': fold+1,
            'accuracy': acc,
            'f1_score': f1
        })

    df = pd.DataFrame(fold_results)
    df.to_csv(f'resultados_kfold_{dataset_type}.csv', index=False)

    print("\n===== RESULTADOS FINAIS DO K-FOLD =====")
    print(df)
    print("\nMédias:")
    print(df.mean())

    # recria o melhor modelo e carrega pesos
    best_model = criar_modelo(
        backbone=backbone,
        num_classes=len(class_names),
        pretrained=False
    ).to(DEVICE)

    best_model.load_state_dict(best_model_state)

    return best_model

In [18]:
path = '/content/dataset/treino_e_validacao'
classes = ['healthy', 'severe']

# MobileNet
mobnet_recplot = run_kfold( path, "F-RecPlot", classes, backbone="mobilenet")

# EfficientNet-B0
effnet_recplot = run_kfold( path, "F-RecPlot", classes, backbone="efficientnet_b0")

# MobileNet
mobnet_orig = run_kfold( path, "originais", classes, backbone="mobilenet")

# EfficientNet-B0
effnet_orig = run_kfold( path, "originais", classes, backbone="efficientnet_b0")

Imagens encontradas em healthy: 92
Imagens encontradas em severe: 92
Total de imagens: 184

FOLD 1/5


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 104MB/s] 


Época 1/20 | TrainAcc 0.7551 | ValAcc 0.5946 | ValF1 0.5469
Época 2/20 | TrainAcc 0.9592 | ValAcc 0.7027 | ValF1 0.6881
Época 3/20 | TrainAcc 1.0000 | ValAcc 0.8378 | ValF1 0.8368
Época 4/20 | TrainAcc 0.9932 | ValAcc 0.8919 | ValF1 0.8918
Época 5/20 | TrainAcc 1.0000 | ValAcc 0.9459 | ValF1 0.9459
Época 6/20 | TrainAcc 1.0000 | ValAcc 0.9730 | ValF1 0.9730
Época 7/20 | TrainAcc 1.0000 | ValAcc 0.9459 | ValF1 0.9459
Época 8/20 | TrainAcc 1.0000 | ValAcc 0.9459 | ValF1 0.9459
Época 9/20 | TrainAcc 1.0000 | ValAcc 0.9459 | ValF1 0.9459
Época 10/20 | TrainAcc 1.0000 | ValAcc 0.9730 | ValF1 0.9730
Época 11/20 | TrainAcc 1.0000 | ValAcc 0.9730 | ValF1 0.9730
Época 12/20 | TrainAcc 1.0000 | ValAcc 0.9730 | ValF1 0.9730
Época 13/20 | TrainAcc 1.0000 | ValAcc 0.9459 | ValF1 0.9459
Época 14/20 | TrainAcc 1.0000 | ValAcc 0.9459 | ValF1 0.9459
Época 15/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 16/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 17/20 | TrainAcc 1.0000 | V

...

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Época 1/20 | TrainAcc 0.6327 | ValAcc 0.5676 | ValF1 0.5067
Época 2/20 | TrainAcc 0.9456 | ValAcc 0.6486 | ValF1 0.6073
Época 3/20 | TrainAcc 0.9864 | ValAcc 0.8919 | ValF1 0.8912
Época 4/20 | TrainAcc 0.9932 | ValAcc 0.9730 | ValF1 0.9730
Época 5/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 6/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 7/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 8/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 9/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 10/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 11/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 12/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 13/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 14/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 15/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 16/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 17/20 | TrainAcc 1.0000 | V

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Época 1/20 | TrainAcc 0.5270 | ValAcc 0.3889 | ValF1 0.3870
Época 2/20 | TrainAcc 0.8986 | ValAcc 0.7222 | ValF1 0.7078
Época 3/20 | TrainAcc 0.9797 | ValAcc 0.7500 | ValF1 0.7333
Época 4/20 | TrainAcc 1.0000 | ValAcc 0.8611 | ValF1 0.8584
Época 5/20 | TrainAcc 0.9932 | ValAcc 0.8889 | ValF1 0.8875
Época 6/20 | TrainAcc 0.9932 | ValAcc 0.8889 | ValF1 0.8875
Época 7/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 8/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 9/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 10/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 11/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 12/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 13/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 14/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 15/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 16/20 | TrainAcc 1.0000 | ValAcc 1.0000 | ValF1 1.0000
Época 17/20 | TrainAcc 1.0000 | V

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


##### Seção de testes

In [31]:
TEST_DIR = '/content/dataset/testes'
CLASSES = ['healthy', 'severe']

In [32]:
class EnsembleTestDataset(Dataset):
    def __init__(self, root_dir, class_names, transform=None):
        self.root = Path(root_dir)
        self.transform = transform
        self.class_names = class_names
        self.data = []

        valid_exts = [".png", ".jpg", ".jpeg", ".tif", ".tiff"]

        for label_idx, class_name in enumerate(class_names):
            path_orig = self.root / class_name / "originais"
            path_rec  = self.root / class_name / "F-RecPlot"

            if not path_orig.exists() or not path_rec.exists():
                print(f"Diretórios não encontrados para classe {class_name}")
                continue

            # cria um dicionário nome_base -> caminho
            rec_dict = {
                p.stem: p
                for p in path_rec.iterdir()
                if p.is_file() and p.suffix.lower() in valid_exts
            }

            orig_files = [
                p for p in path_orig.iterdir()
                if p.is_file() and p.suffix.lower() in valid_exts
            ]

            for orig_path in orig_files:
                key = orig_path.stem  # nome sem extensão

                if key in rec_dict:
                    self.data.append({
                        "path_orig": str(orig_path),
                        "path_rec": str(rec_dict[key]),
                        "label": label_idx
                    })
                else:
                    print(f"Aviso: Par não encontrado para {orig_path.name}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        img_orig = Image.open(item["path_orig"]).convert("RGB")
        img_rec  = Image.open(item["path_rec"]).convert("RGB")
        label = item["label"]

        if self.transform:
            img_orig = self.transform(img_orig)
            img_rec  = self.transform(img_rec)

        return img_orig, img_rec, label


In [33]:
def carregar_modelo(backbone, num_classes, path_weights):
    print(f"Carregando {backbone} de {path_weights}...")
    backbone = backbone.lower()
    if backbone == "mobilenet":
        model = models.mobilenet_v2(pretrained=False)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif backbone == "efficientnet_b0":
        model = models.efficientnet_b0(pretrained=False)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    # Carrega os pesos treinados
    try:
        model.load_state_dict(torch.load(path_weights, map_location=DEVICE))
    except FileNotFoundError:
        print(f"ERRO: Arquivo {path_weights} não encontrado! Treine o modelo antes.")
        return None

    model.to(DEVICE)
    model.eval()
    return model

Criar o dataset e o loader

In [34]:
test_dataset = EnsembleTestDataset(TEST_DIR, CLASSES, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Total de pares de imagens para teste: {len(test_dataset)}")

Total de pares de imagens para teste: 44


Carregar os 4 possíveis modelos

In [35]:
# 1. MobileNet - RecPlot
mob_rec = carregar_modelo("mobilenet", 2, "melhor_modelo_mobilenet_F-RecPlot.pth")
# 2. MobileNet - Originais
mob_orig = carregar_modelo("mobilenet", 2, "melhor_modelo_mobilenet_originais.pth")
# 3. EfficientNet - RecPlot
eff_rec = carregar_modelo("efficientnet_b0", 2, "melhor_modelo_efficientnet_b0_F-RecPlot.pth")
# 4. EfficientNet - Originais
eff_orig = carregar_modelo("efficientnet_b0", 2, "melhor_modelo_efficientnet_b0_originais.pth")


Carregando mobilenet de melhor_modelo_mobilenet_F-RecPlot.pth...
Carregando mobilenet de melhor_modelo_mobilenet_originais.pth...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Carregando efficientnet_b0 de melhor_modelo_efficientnet_b0_F-RecPlot.pth...
Carregando efficientnet_b0 de melhor_modelo_efficientnet_b0_originais.pth...


In [36]:
y_true = []

probs_cenario_1 = []
probs_cenario_2 = []
probs_cenario_3 = []
probs_cenario_4 = []

In [37]:
with torch.no_grad():
    for imgs_orig, imgs_rec, labels in test_loader:
        imgs_orig, imgs_rec = imgs_orig.to(DEVICE), imgs_rec.to(DEVICE)


        # MobileNet RecPlot
        out_mob_rec = F.softmax(mob_rec(imgs_rec), dim=1)
        # MobileNet Original
        out_mob_orig = F.softmax(mob_orig(imgs_orig), dim=1)
        # EfficientNet RecPlot
        out_eff_rec = F.softmax(eff_rec(imgs_rec), dim=1)
        # EfficientNet Original
        out_eff_orig = F.softmax(eff_orig(imgs_orig), dim=1)

        # Guardar labels reais
        y_true.extend(labels.numpy())

        # Cenário 1: 2 CNNs (Mobile) -> Entrada: Original + RecPlot
        soma_c1 = out_mob_orig + out_mob_rec
        probs_cenario_1.extend(torch.argmax(soma_c1, dim=1).cpu().numpy())

        # Cenário 2: 1 Mobile + 1 EffNet -> Entrada: Original + RecPlot
        soma_c2 = out_mob_orig + out_eff_rec
        probs_cenario_2.extend(torch.argmax(soma_c2, dim=1).cpu().numpy())

        # Cenário 3: 1 Mobile + 1 EffNet -> Entrada: Só Original
        soma_c3 = out_mob_orig + out_eff_orig
        probs_cenario_3.extend(torch.argmax(soma_c3, dim=1).cpu().numpy())

        # Cenário 4: 1 Mobile + 1 EffNet -> Entrada: Só RecPlot
        soma_c4 = out_mob_rec + out_eff_rec
        probs_cenario_4.extend(torch.argmax(soma_c4, dim=1).cpu().numpy())

In [38]:
def mostrar_metricas(nome_cenario, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n>>> {nome_cenario}")
    print(f"    Acurácia: {acc:.4f}")
    print(f"    F1 Score: {f1:.4f}")
    print(f"    Matriz de Confusão:\n{cm}")

print("\n================ RELATÓRIO FINAL DE TESTE ================")
mostrar_metricas("Cenário 1: 2 MobileNets (Original + RecPlot)", y_true, probs_cenario_1)
mostrar_metricas("Cenário 2: MobileNet + EfficientNet (Original + RecPlot)", y_true, probs_cenario_2)
mostrar_metricas("Cenário 3: MobileNet + EfficientNet (Apenas Original)", y_true, probs_cenario_3)
mostrar_metricas("Cenário 4: MobileNet + EfficientNet (Apenas RecPlot)", y_true, probs_cenario_4)


================ RELATÓRIO FINAL DE TESTE ================

>>> Cenário 1: 2 MobileNets (Original + RecPlot)
    Acurácia: 0.9773
    F1 Score: 0.9773
    Matriz de Confusão:
[[21  1]
 [ 0 22]]

>>> Cenário 2: MobileNet + EfficientNet (Original + RecPlot)
    Acurácia: 0.9545
    F1 Score: 0.9545
    Matriz de Confusão:
[[20  2]
 [ 0 22]]

>>> Cenário 3: MobileNet + EfficientNet (Apenas Original)
    Acurácia: 0.8636
    F1 Score: 0.8611
    Matriz de Confusão:
[[16  6]
 [ 0 22]]

>>> Cenário 4: MobileNet + EfficientNet (Apenas RecPlot)
    Acurácia: 0.9773
    F1 Score: 0.9773
    Matriz de Confusão:
[[21  1]
 [ 0 22]]
